In [ ]:
import os
import math
import h5py
import numpy as np
import pickle
import torch
import torch.nn as nn
import tensorflow as tf
from tensorflow.keras.models import load_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

loadPath = '/home/sz4544/EEG-motor-imagery-main/project/'

# ---------------------------
# load latent train
# ---------------------------
f = h5py.File(os.path.join(loadPath, "train1800_latent.h5"), "r")
z_train = f["latent"][:]
y_train = f["tasks"][:]
f.close()

y_train = y_train.astype(np.int64)
if np.array_equal(np.unique(y_train), np.array([1,2,3,4])):
    y_train = y_train - 1

latent_dim = z_train.shape[1]

# ---------------------------
# diffusion model definition
# ---------------------------
class SinusoidalPosEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, t):
        half_dim = self.dim // 2
        emb = math.log(10000) / max(half_dim - 1, 1)
        emb = torch.exp(torch.arange(half_dim, device=t.device) * -emb)
        emb = t[:, None] * emb[None, :]
        return torch.cat((emb.sin(), emb.cos()), dim=1)

class LatentDiffusionMLP(nn.Module):
    def __init__(self, latent_dim=128, n_classes=4, hidden=256, time_dim=128):
        super().__init__()
        self.time_emb = SinusoidalPosEmb(time_dim)
        self.time_mlp = nn.Sequential(
            nn.Linear(time_dim, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden)
        )
        self.class_emb = nn.Embedding(n_classes, hidden)

        self.net = nn.Sequential(
            nn.Linear(latent_dim + hidden, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden),
            nn.SiLU(),
            nn.Linear(hidden, latent_dim)
        )

    def forward(self, z, t, y):
        t_emb = self.time_mlp(self.time_emb(t))
        c_emb = self.class_emb(y)
        cond = t_emb + c_emb
        inp = torch.cat([z, cond], dim=1)
        return self.net(inp)

model = LatentDiffusionMLP(latent_dim=latent_dim).to(device)
model.load_state_dict(torch.load(os.path.join(loadPath, "models", "latent_diffusion.pt"), map_location=device))
model.eval()

T = 500
betas = torch.linspace(1e-4, 0.02, T, device=device)
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)
alphas_cumprod_prev = torch.cat([torch.tensor([1.0], device=device), alphas_cumprod[:-1]], dim=0)
posterior_variance = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)
posterior_variance = torch.clamp(posterior_variance, min=1e-12)

def sample_one(class_id):
    z = torch.randn(1, latent_dim, device=device)
    y = torch.tensor([class_id], device=device, dtype=torch.long)

    for t in reversed(range(T)):
        t_tensor = torch.tensor([t], device=device, dtype=torch.float32)
        with torch.no_grad():
            eps = model(z, t_tensor, y)

        alpha_t = alphas[t]
        alpha_bar_t = alphas_cumprod[t]
        beta_t = betas[t]

        z = (z - (beta_t / torch.sqrt(1.0 - alpha_bar_t)) * eps) / torch.sqrt(alpha_t)

        if t > 0:
            noise = torch.randn_like(z)
            z = z + torch.sqrt(posterior_variance[t]) * noise

    return z.detach().cpu().numpy()[0]

# ---------------------------
# load decoder + scaler
# ---------------------------
# ---------------------------
# load decoder + scaler
# ---------------------------
autoencoder = load_model(os.path.join(loadPath, "models", "latent_autoencoder.keras"))

# rebuild decoder from the layer right AFTER latent_vector
latent_inputs = tf.keras.Input(shape=(latent_dim,), name="decoder_input")

latent_idx = None
for i, layer in enumerate(autoencoder.layers):
    if layer.name == "latent_vector":
        latent_idx = i
        break


if latent_idx is None:
    raise ValueError("Could not find layer named 'latent_vector' in autoencoder.")

x = latent_inputs
for layer in autoencoder.layers[latent_idx + 1:]:
    x = layer(x)

decoder = tf.keras.Model(latent_inputs, x, name="decoder")

print(decoder.summary())

with open(os.path.join(loadPath, "models", "latent_autoencoder_scaler.pkl"), "rb") as f:
    scaler = pickle.load(f)

# ---------------------------
# generate small amount first
# ---------------------------
samples_per_class = 100

all_latents = []
all_labels = []

for cls in range(4):
    print("class", cls)
    for i in range(samples_per_class):
        if i % 20 == 0:
            print("sample", i)
        z = sample_one(cls)
        all_latents.append(z)
        all_labels.append(cls)

fake_z = np.stack(all_latents).astype(np.float32)
fake_y = np.array(all_labels).astype(np.int64)

print("fake_z:", fake_z.shape)
print("fake_y:", fake_y.shape)

# decode to standardized EEG
fake_x_scaled = decoder.predict(fake_z, batch_size=64)
print("fake_x_scaled:", fake_x_scaled.shape)

# inverse scale back to raw EEG
flat = fake_x_scaled.reshape((-1, 64))
fake_x_raw_flat = scaler.inverse_transform(flat)
fake_x_raw = fake_x_raw_flat.reshape((-1, 640, 64, 1)).squeeze(-1).astype(np.float32)

print("fake_x_raw:", fake_x_raw.shape)
print("fake sample stats:", np.mean(fake_x_raw[0]), np.std(fake_x_raw[0]))

# ---------------------------
# load original raw train and save augmented file
# ---------------------------
f = h5py.File(os.path.join(loadPath, "train1800_raw_EEG.h5"), "r")
orig_x = f["data"][:]
orig_y = f["tasks"][:]
orig_subjects = f["subjects"][:]
f.close()

# assign synthetic subjects = -1
fake_subjects = np.full((len(fake_y),), -1)

x_all = np.concatenate([orig_x, fake_x_raw], axis=0)
y_all = np.concatenate([orig_y, fake_y], axis=0)
subjects_all = np.concatenate([orig_subjects, fake_subjects], axis=0)

save_path = os.path.join(loadPath, "train1800_latentdiff_aug.h5")
f = h5py.File(save_path, "w")
f.create_dataset("data", data=x_all)
f.create_dataset("tasks", data=y_all)
f.create_dataset("subjects", data=subjects_all)
f.close()

print("saved:", save_path)
print("final train shape:", x_all.shape, y_all.shape)

I0000 00:00:1774223624.181274  124880 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


cuda


I0000 00:00:1774223626.277618  124880 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22206 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:41:00.0, compute capability: 8.9
I0000 00:00:1774223626.278334  124880 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 22253 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:61:00.0, compute capability: 8.9


Model: "decoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ decoder_input (InputLayer)      │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 40960)          │     5,283,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_3 (ReLU)                  │ (None, 40960)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 80, 8, 64)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose                │ (None, 160, 16, 32)    │        92,192 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 160, 16, 32)    │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_4 (ReLU)                  │ (None, 160, 16, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_1              │ (None, 320, 32, 16)    │        23,056 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 320, 32, 16)    │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_5 (ReLU)                  │ (None, 320, 32, 16)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_2              │ (None, 640, 64, 8)     │         5,768 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 640, 64, 8)     │            32 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_6 (ReLU)                  │ (None, 640, 64, 8)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 640, 64, 1)     │            73 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

...

None
class 0
sample 0
sample 20
sample 40
sample 60
sample 80
class 1
sample 0
sample 20
sample 40
sample 60
sample 80
class 2
sample 0
sample 20
sample 40
sample 60
sample 80
class 3
sample 0
sample 20
sample 40
sample 60
sample 80
fake_z: (400, 128)
fake_y: (400,)


I0000 00:00:1774223707.158000  125035 service.cc:153] XLA service 0x7cf08802fbd0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1774223707.158037  125035 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 4090, Compute Capability 8.9 (Driver: 12.4.0; Runtime: 12.8.0; Toolkit: 12.5.0; DNN: 9.10.2)
I0000 00:00:1774223707.158050  125035 service.cc:161]   StreamExecutor [1]: NVIDIA GeForce RTX 4090, Compute Capability 8.9 (Driver: 12.4.0; Runtime: 12.8.0; Toolkit: 12.5.0; DNN: 9.10.2)
I0000 00:00:1774223707.171074  125035 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1774223707.215878  125035 cuda_dnn.cc:461] Loaded cuDNN version 91002


1/7 ━━━━━━━━━━━━━━━━━━━━ 9s 2s/step

I0000 00:00:1774223708.619952  125035 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 127ms/step
fake_x_scaled: (400, 640, 64, 1)
fake_x_raw: (400, 640, 64)
fake sample stats: -6.1345134e-05 3.3226323e-05
saved: /home/sz4544/EEG-motor-imagery-main/project/train1800_latentdiff_aug.h5
final train shape: (2200, 640, 64) (2200,)
